In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/ecommerce_customer_behavior.csv")

In [3]:
# Standardize column names — remove spaces, lowercase
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
print("Cleaned columns:", df.columns.tolist())

Cleaned columns: ['customer_id', 'gender', 'age', 'city', 'membership_type', 'total_spend', 'items_purchased', 'average_rating', 'discount_applied', 'days_since_last_purchase', 'satisfaction_level']


In [4]:
def assign_s1_s2_label(row):
    score = 0

    # ── Gender signal ─────────────────────────────────────
    # Small signal — research shows slight difference
    gender = str(row.get('gender', '')).lower()
    if gender == 'female':
        score += 1   # Slightly more hedonic purchasing

    # ── Age signal ────────────────────────────────────────
    # Strongest signal — young = impulsive, older = deliberate
    age = pd.to_numeric(row.get('age', 35), errors='coerce')
    if pd.isna(age):
        pass
    elif age < 25:
        score += 3   # Gen Z — very impulsive
    elif age < 35:
        score += 2   # Millennials — moderately impulsive
    elif age < 50:
        score += 0   # Gen X — neutral
    else:
        score -= 2   # Boomers — deliberate, research-driven

    # ── Membership type (economic status proxy) ───────────
    # Bronze = low spend = impulse on small items
    # Gold = high spend = deliberate on quality
    membership = str(row.get('membership_type', '')).lower()
    if 'bronze' in membership:
        score += 1   # Lower tier = budget-conscious impulse
    elif 'silver' in membership:
        score += 0   # Middle = neutral
    elif 'gold' in membership:
        score -= 2   # Gold = high-value deliberate buyers

    # ── Total spend signal (economic status) ─────────────
    spend = pd.to_numeric(row.get('total_spend', 200), errors='coerce')
    if pd.isna(spend):
        pass
    elif spend < 200:
        score += 2   # Low total = impulse small purchases
    elif spend < 600:
        score += 0   # Medium = neutral
    else:
        score -= 2   # High total = deliberate big purchases

    # ── Items purchased (behavior signal) ────────────────
    # Many small items = impulsive browsing (System 1)
    # Few large items = deliberate selection (System 2)
    items = pd.to_numeric(row.get('items_purchased', 5), errors='coerce')
    if pd.isna(items):
        pass
    elif items >= 10:
        score += 2   # High item count = impulse buyer
    elif items >= 5:
        score += 1
    elif items <= 2:
        score -= 1   # Very few = selective deliberate buyer

    # ── Days since last purchase (recency/frequency) ──────
    # Frequent buyer (low days) = habitual/impulsive
    # Infrequent (high days) = deliberate planned
    days = pd.to_numeric(
        row.get('days_since_last_purchase', 30), errors='coerce'
    )
    if pd.isna(days):
        pass
    elif days <= 10:
        score += 2   # Very frequent = impulsive System 1
    elif days <= 30:
        score += 1
    elif days >= 60:
        score -= 1   # Infrequent = deliberate System 2

    # ── Discount applied (price sensitivity) ─────────────
    # Discount seekers = budget conscious → more rational
    discount = str(row.get('discount_applied', 'no')).lower()
    if discount == 'yes':
        score -= 1   # Discount seeking = System 2 behavior

    # ── Convert score to label ────────────────────────────
    if score >= 2:
        return 1      # System 1 — emotional/impulsive
    elif score <= -1:
        return 0      # System 2 — rational/deliberate
    else:
        return None   # Ambiguous — exclude from training

In [5]:
# Apply labeling
df['cognitive_label'] = df.apply(assign_s1_s2_label, axis=1)

# Show distribution
total     = len(df)
labeled   = df['cognitive_label'].notna().sum()
s1_count  = (df['cognitive_label'] == 1).sum()
s2_count  = (df['cognitive_label'] == 0).sum()
ambiguous = df['cognitive_label'].isna().sum()

print(f"\n{'='*45}")
print(f"LABELING RESULTS")
print(f"{'='*45}")
print(f"Total rows        : {total}")
print(f"Labeled rows      : {labeled}")
print(f"  System 1 (S1)   : {s1_count}")
print(f"  System 2 (S2)   : {s2_count}")
print(f"Ambiguous excluded: {ambiguous}")
print(f"Balance ratio     : {min(s1_count,s2_count)/max(s1_count,s2_count):.2f}")
print(f"{'='*45}")



LABELING RESULTS
Total rows        : 350
Labeled rows      : 284
  System 1 (S1)   : 283
  System 2 (S2)   : 1
Ambiguous excluded: 66
Balance ratio     : 0.00


In [6]:
df_labeled = df[df['cognitive_label'].notna()].copy()
df_labeled.to_csv("../data/processed/demographic_labeled.csv", index=False)
print("\nSaved → ../data/processed/demographic_labeled.csv")


Saved → ../data/processed/demographic_labeled.csv


In [7]:
# Quick validation — check if labels align with expectations
print("\nLABEL VALIDATION")
print("-" * 40)

# S1 group should be: younger, more items, lower spend, recent purchases
s1_group = df_labeled[df_labeled.cognitive_label == 1]
s2_group = df_labeled[df_labeled.cognitive_label == 0]

metrics = {
    'avg_age':          ('age', 'mean'),
    'avg_items':        ('items_purchased', 'mean'),
    'avg_spend':        ('total_spend', 'mean'),
    'avg_days':         ('days_since_last_purchase', 'mean'),
}

for label_name, (col, func) in metrics.items():
    col_clean = col.lower().replace(' ', '_')
    if col_clean in df_labeled.columns:
        s1_val = getattr(s1_group[col_clean], func)()
        s2_val = getattr(s2_group[col_clean], func)()
        direction = "✅" if (
            (label_name in ['avg_age','avg_spend','avg_days'] and s2_val > s1_val) or
            (label_name == 'avg_items' and s1_val > s2_val)
        ) else "⚠️ CHECK"
        print(f"{label_name:<20} S1={s1_val:.1f} | S2={s2_val:.1f} {direction}")


LABEL VALIDATION
----------------------------------------
avg_age              S1=34.1 | S2=36.0 ✅
avg_items            S1=11.7 | S2=19.0 ⚠️ CHECK
avg_spend            S1=772.4 | S2=1420.8 ✅
avg_days             S1=28.6 | S2=11.0 ⚠️ CHECK
